<a href="https://colab.research.google.com/github/ewawegrzynek86-crypto/building-ai-law-hackathon/blob/main/Deterministic_Legal_Intake_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This system combines LLM-based structured extraction with a deterministic validation layer. It prevents blind automation by introducing rule-based SOL calculation, red-flag detection, and confidence-based routing.

Three possible outcomes:

READY_FOR_AUTOMATION

READY_REVIEW_REQUIRED

BLOCKED_FOR_AUTOMATION

police_report_text = """
On March 12, 2023, John Miller was driving eastbound on Main Street in Brooklyn, NY
when another vehicle driven by Michael Torres ran a red light and struck the driver’s side.
The client reports neck and lower back pain. Police report number NYPD-23-458771.
"""

In [ ]:
print("Simulated LLM extraction layer → structured JSON output")

Simulated LLM extraction layer → structured JSON output


In [ ]:
import re
from datetime import datetime

VALID_ROLES = ["Driver", "Passenger", "Pedestrian", "Unknown"]
# Rozszerzona lista subiektywnych słów dla bezpieczeństwa
SUBJECTIVE_WORDS = ["guilty", "fault", "probably", "likely", "appears", "seems", "responsible"]
CONFIDENCE_THRESHOLD = 60

def is_valid_iso(date_string):
    if not date_string or not isinstance(date_string, str):
        return False
    pattern = r"^\d{4}-\d{2}-\d{2}$"
    if not re.match(pattern, date_string):
        return False
    try:
        datetime.strptime(date_string, "%Y-%m-%d")
        return True
    except ValueError:
        return False

def calculate_sol(accident_date):
    """Bezpieczne wyliczanie SOL nawet dla 29 lutego."""
    dt = datetime.strptime(accident_date, "%Y-%m-%d")
    try:
        return dt.replace(year=dt.year + 8).strftime("%Y-%m-%d")
    except ValueError:
        # Obsługa 29 lutego -> przesuwa na 28 lutego za 8 lat
        return dt.replace(year=dt.year + 8, day=28).strftime("%Y-%m-%d")

def validate_case(data):
    # Pobieramy bazowe wartości lub inicjalizujemy puste
    final_confidence = data.get("base_confidence", 50)
    # Łączymy flagi od AI z tymi, które zaraz znajdzie nasz kod
    red_flags = data.get("red_flags", [])
    missing = data.get("missing_critical_fields", [])

    accident_date = data.get("accident_date")

    # 1. WALIDACJA DATY
    if not is_valid_iso(accident_date):
        if "accident_date" not in missing:
            missing.append("accident_date")
        final_confidence -= 30 # Wyższa kara za brak daty (klucz do SOL)
        sol_date = None
    else:
        accident_dt = datetime.strptime(accident_date, "%Y-%m-%d")
        today = datetime.today()

        if accident_dt > today:
            red_flags.append("Accident date in future")
            final_confidence -= 20

        if accident_dt.year < 1990: # Realistyczna granica dla spraw cywilnych
            red_flags.append("Suspicious accident year (possible OCR error)")
            final_confidence -= 15

        sol_date = calculate_sol(accident_date)

    # 2. WALIDACJA ROLI (ENUM)
    role = data.get("client_role")
    if role not in VALID_ROLES:
        data["client_role"] = "Unknown"
        red_flags.append(f"Invalid role '{role}' normalized to Unknown")
        final_confidence -= 5

    # 3. STRAŻNIK SUBIEKTYWIZMU (Liability Guard)
    liability = data.get("liability_indicator")
    if liability:
        # Szukamy całych słów, żeby uniknąć błędów (np. "fault" w "default")
        for word in SUBJECTIVE_WORDS:
            if re.search(rf'\b{word}\b', liability.lower()):
                red_flags.append(f"Subjective word '{word}' detected in liability")
                data["liability_indicator"] = None # Czyścimy pole dla bezpieczeństwa
                final_confidence -= 15
                break

    # 4. KRYTYCZNE BRAKI (GATEKEEPER)
    for field in ["client_name", "accident_date", "accident_location"]:
        if not data.get(field):
            if field not in missing:
                missing.append(field)

    # 5. FINALNE PRZELICZENIE (Penalty Layer)
    # Kara za każdą flagę, ale nie schodzimy poniżej 0
    final_confidence -= (10 * len(red_flags))
    data["final_confidence"] = max(0, final_confidence)

    data["missing_critical_fields"] = missing
    data["red_flags"] = red_flags
    data["sol_date"] = sol_date

    # Logika Statusu (Twoja była świetna, zostawiamy!)
    critical_missing = any(f in missing for f in ["client_name", "accident_date", "accident_location"])

    if critical_missing:
        data["status"] = "BLOCKED_FOR_AUTOMATION"
    elif data["final_confidence"] < CONFIDENCE_THRESHOLD:
        data["status"] = "READY_REVIEW_REQUIRED"
    else:
        data["status"] = "READY_FOR_AUTOMATION"

    return data

In [ ]:
import json
import copy

def run_full_pipeline(simulated_llm_output):
    # Tworzymy kopię, żeby nie zmieniać oryginału
    raw_input = copy.deepcopy(simulated_llm_output)

    print("\n" + "="*50)
    print("🚀 STEP 1: RAW LLM EXTRACTION (The 'Chaos' Layer)")
    print("="*50)
    print(json.dumps(raw_input, indent=4))

    print("\n" + "="*50)
    print("🛡️ STEP 2: VALIDATION LAYER (The 'Intelligence' Layer)")
    print("="*50)

    # Walidacja
    validated = validate_case(raw_input)

    # Emoji statusu
    if validated["status"] == "READY_FOR_AUTOMATION":
        status_emoji = "✅"
    elif validated["status"] == "BLOCKED_FOR_AUTOMATION":
        status_emoji = "❌"
    else:
        status_emoji = "⚠️"

    print(f"SYSTEM STATUS: {status_emoji} {validated['status']}")
    print(f"FINAL CONFIDENCE SCORE: {validated['final_confidence']}/100")
    print(f"CALCULATED SOL DATE: {validated['sol_date']}")

    # Tylko dla READY
    if validated["status"] == "READY_FOR_AUTOMATION":
        print("→ Ready for Clio Matter Creation")

    print("\n🚩 DETECTED ISSUES:")
    print(f"- Red Flags: {validated['red_flags'] if validated['red_flags'] else 'None'}")
    print(f"- Missing Critical: {validated['missing_critical_fields'] if validated['missing_critical_fields'] else 'None'}")

    print("\n" + "="*50)
    print("📋 FINAL CASE SNAPSHOT (Ready for Clio)")
    print("="*50)
    print(json.dumps(validated, indent=4))

    return validated

In [ ]:
ready_case = {
    "client_name": "John Miller",
    "accident_date": "2023-01-10",
    "accident_time": "15:45",
    "accident_location": "Main Street, Brooklyn, NY",
    "other_driver_name": "Michael Torres",
    "insurance_company": "Liberty Mutual",
    "police_report_number": "NYPD-23-458771",
    "client_role": "Driver",
    "reported_injuries": ["neck pain", "lower back pain"],
    "liability_indicator": "red-light violation",
    "base_confidence": 92,
    "missing_critical_fields": [],
    "red_flags": []
}

result_ready = run_full_pipeline(ready_case)


🚀 STEP 1: RAW LLM EXTRACTION (The 'Chaos' Layer)
{
    "client_name": "John Miller",
    "accident_date": "2023-01-10",
    "accident_time": "15:45",
    "accident_location": "Main Street, Brooklyn, NY",
    "other_driver_name": "Michael Torres",
    "insurance_company": "Liberty Mutual",
    "police_report_number": "NYPD-23-458771",
    "client_role": "Driver",
    "reported_injuries": [
        "neck pain",
        "lower back pain"
    ],
    "liability_indicator": "red-light violation",
    "base_confidence": 92,
    "missing_critical_fields": [],
    "red_flags": []
}

🛡️ STEP 2: VALIDATION LAYER (The 'Intelligence' Layer)
SYSTEM STATUS: ✅ READY_FOR_AUTOMATION
FINAL CONFIDENCE SCORE: 92/100
CALCULATED SOL DATE: 2031-01-10
→ Ready for Clio Matter Creation

🚩 DETECTED ISSUES:
- Red Flags: None
- Missing Critical: None

📋 FINAL CASE SNAPSHOT (Ready for Clio)
{
    "client_name": "John Miller",
    "accident_date": "2023-01-10",
    "accident_time": "15:45",
    "accident_location": 

In [ ]:
blocked_case = {
    "client_name": "John Miller",
    "accident_date": None,
    "accident_time": None,
    "accident_location": "Main Street, Brooklyn, NY",
    "other_driver_name": "Michael Torres",
    "insurance_company": "Liberty Mutual",
    "police_report_number": None,
    "client_role": "Driver",
    "reported_injuries": ["back pain"],
    "liability_indicator": None,
    "base_confidence": 80,
    "missing_critical_fields": [],
    "red_flags": []
}

result_blocked = run_full_pipeline(blocked_case)


🚀 STEP 1: RAW LLM EXTRACTION (The 'Chaos' Layer)
{
    "client_name": "John Miller",
    "accident_date": null,
    "accident_time": null,
    "accident_location": "Main Street, Brooklyn, NY",
    "other_driver_name": "Michael Torres",
    "insurance_company": "Liberty Mutual",
    "police_report_number": null,
    "client_role": "Driver",
    "reported_injuries": [
        "back pain"
    ],
    "liability_indicator": null,
    "base_confidence": 80,
    "missing_critical_fields": [],
    "red_flags": []
}

🛡️ STEP 2: VALIDATION LAYER (The 'Intelligence' Layer)
SYSTEM STATUS: ❌ BLOCKED_FOR_AUTOMATION
FINAL CONFIDENCE SCORE: 50/100
CALCULATED SOL DATE: None

🚩 DETECTED ISSUES:
- Red Flags: None
- Missing Critical: ['accident_date']

📋 FINAL CASE SNAPSHOT (Ready for Clio)
{
    "client_name": "John Miller",
    "accident_date": null,
    "accident_time": null,
    "accident_location": "Main Street, Brooklyn, NY",
    "other_driver_name": "Michael Torres",
    "insurance_company": "Libe

In [ ]:
review_case = {
    "client_name": "John Miller",
    "accident_date": "2035-05-12",  # future date
    "accident_time": "14:00",
    "accident_location": "Main Street, Brooklyn, NY",
    "other_driver_name": "Michael Torres",
    "insurance_company": "Liberty Mutual",
    "police_report_number": "NYPD-23-458771",
    "client_role": "Driver",
    "reported_injuries": ["neck pain"],
    "liability_indicator": "rear-end collision",
    "base_confidence": 70,
    "missing_critical_fields": [],
    "red_flags": []
}

result_review = run_full_pipeline(review_case)


🚀 STEP 1: RAW LLM EXTRACTION (The 'Chaos' Layer)
{
    "client_name": "John Miller",
    "accident_date": "2035-05-12",
    "accident_time": "14:00",
    "accident_location": "Main Street, Brooklyn, NY",
    "other_driver_name": "Michael Torres",
    "insurance_company": "Liberty Mutual",
    "police_report_number": "NYPD-23-458771",
    "client_role": "Driver",
    "reported_injuries": [
        "neck pain"
    ],
    "liability_indicator": "rear-end collision",
    "base_confidence": 70,
    "missing_critical_fields": [],
    "red_flags": []
}

🛡️ STEP 2: VALIDATION LAYER (The 'Intelligence' Layer)
SYSTEM STATUS: ⚠️ READY_REVIEW_REQUIRED
FINAL CONFIDENCE SCORE: 40/100
CALCULATED SOL DATE: 2043-05-12

🚩 DETECTED ISSUES:
- Red Flags: ['Accident date in future']
- Missing Critical: None

📋 FINAL CASE SNAPSHOT (Ready for Clio)
{
    "client_name": "John Miller",
    "accident_date": "2035-05-12",
    "accident_time": "14:00",
    "accident_location": "Main Street, Brooklyn, NY",
    "ot

SYSTEM ARCHITECTURE:

Client Narrative
        ↓
LLM Structured Extraction (Simulated)
        ↓
Deterministic Validation Layer
        ↓
Automation Routing Decision

In [ ]:
print("=== DEMO SUMMARY ===")
print("AI performs structured extraction.")
print("Validation layer enforces legal safety rules.")
print("System routes cases to:")
print("- READY_FOR_AUTOMATION")
print("- BLOCKED_FOR_AUTOMATION")
print("- READY_REVIEW_REQUIRED")
print("")
print("This prevents malpractice risk and blind LLM automation.")

=== DEMO SUMMARY ===
AI performs structured extraction.
Validation layer enforces legal safety rules.
System routes cases to:
- READY_FOR_AUTOMATION
- BLOCKED_FOR_AUTOMATION
- READY_REVIEW_REQUIRED

This prevents malpractice risk and blind LLM automation.


In [ ]:
def generate_retainer(case):
    if case["status"] != "READY_FOR_AUTOMATION":
        return "Case not eligible for automated retainer."

    return f"""
    RETAINER AGREEMENT

    Client: {case['client_name']}
    Accident Date: {case['accident_date']}
    Location: {case['accident_location']}
    Opposing Driver: {case['other_driver_name']}

    This firm agrees to represent the client in a personal injury claim
    arising from the above incident.
    """

In [ ]:
def generate_client_email(case):

    if case["status"] != "READY_FOR_AUTOMATION":
        return "Case not eligible for automated email generation."

    return f"""
SUBJECT: Personal Injury Representation – Next Steps

Dear {case['client_name']},

Thank you for contacting our firm regarding your accident on {case['accident_date']}
at {case['accident_location']}.

Based on the information provided, we are prepared to move forward
with representation concerning your personal injury claim.

Attached you will find the retainer agreement outlining the terms of representation.
Please review, sign, and return at your earliest convenience.

If you have any questions, do not hesitate to contact our office.

Sincerely,
Legal Intake Team
"""

In [ ]:
print(generate_client_email(result_ready))


SUBJECT: Personal Injury Representation – Next Steps

Dear John Miller,

Thank you for contacting our firm regarding your accident on 2023-01-10
at Main Street, Brooklyn, NY.

Based on the information provided, we are prepared to move forward
with representation concerning your personal injury claim.

Attached you will find the retainer agreement outlining the terms of representation.
Please review, sign, and return at your earliest convenience.

If you have any questions, do not hesitate to contact our office.

Sincerely,
Legal Intake Team



In [ ]:
print(generate_retainer(result_ready))


    RETAINER AGREEMENT

    Client: John Miller
    Accident Date: 2023-01-10
    Location: Main Street, Brooklyn, NY
    Opposing Driver: Michael Torres

    This firm agrees to represent the client in a personal injury claim
    arising from the above incident.
    


In [ ]:
def full_automation(case):

    print("=== AUTOMATION LAYER ===")

    if case["status"] != "READY_FOR_AUTOMATION":
        print("Automation blocked.")
        print("Status:", case["status"])
        return

    print("\nGenerating Retainer...\n")

    retainer = f"""
    RETAINER AGREEMENT

    Client: {case['client_name']}
    Accident Date: {case['accident_date']}
    Location: {case['accident_location']}
    Opposing Driver: {case['other_driver_name']}

    This firm agrees to represent the client in a personal injury claim
    arising from the above incident.
    """

    print(retainer)

    print("\nGenerating Client Email...\n")

    email = f"""
    SUBJECT: Personal Injury Representation – Next Steps

    Dear {case['client_name']},

    Thank you for contacting our firm regarding your accident on {case['accident_date']}
    at {case['accident_location']}.

    Based on the validated intake information, we are prepared to move forward
    with representation concerning your personal injury claim.

    Please review the attached retainer agreement and return a signed copy.

    Sincerely,
    Legal Intake Team
    """

    print(email)

In [ ]:
full_automation(result_ready)

=== AUTOMATION LAYER ===

Generating Retainer...


    RETAINER AGREEMENT

    Client: John Miller
    Accident Date: 2023-01-10
    Location: Main Street, Brooklyn, NY
    Opposing Driver: Michael Torres

    This firm agrees to represent the client in a personal injury claim
    arising from the above incident.
    

Generating Client Email...


    SUBJECT: Personal Injury Representation – Next Steps

    Dear John Miller,

    Thank you for contacting our firm regarding your accident on 2023-01-10
    at Main Street, Brooklyn, NY.

    Based on the validated intake information, we are prepared to move forward
    with representation concerning your personal injury claim.

    Please review the attached retainer agreement and return a signed copy.

    Sincerely,
    Legal Intake Team
    


In [ ]:
full_automation(result_blocked)

=== AUTOMATION LAYER ===
Automation blocked.
Status: BLOCKED_FOR_AUTOMATION


In [ ]:
full_automation(result_review)

=== AUTOMATION LAYER ===
Automation blocked.
Status: READY_REVIEW_REQUIRED


In [ ]:
!pip install reportlab

In [ ]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib import colors
from reportlab.lib.units import inch
from reportlab.lib.pagesizes import LETTER

def generate_retainer_pdf(data, filename="Retainer.pdf"):
    doc = SimpleDocTemplate(filename, pagesize=LETTER)
    elements = []

    styles = getSampleStyleSheet()
    normal_style = styles["Normal"]

    text_content = f"""
    PERSONAL INJURY CONTINGENCY FEE AGREEMENT

    This Retainer Agreement is entered into between:

    Law Firm: Andrew Richards
    Client: {data['client_name']}

    The Law Firm agrees to represent Client in connection with a motor vehicle accident
    that occurred on {data['accident_date']} at {data['accident_location']} involving
    {data['other_driver_name']}.

    The Law Firm’s fee shall be 33% of the gross amount recovered.

    Statute of Limitations: {data['sol_date']}
    """

    elements.append(Paragraph(text_content, normal_style))
    elements.append(Spacer(1, 0.5 * inch))

    doc.build(elements)

    return filename

In [ ]:
pdf_path = generate_retainer_pdf(result_ready)
pdf_path

'Retainer.pdf'

In [ ]:
from google.colab import files
files.download("Retainer.pdf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from datetime import datetime

def get_booking_link():
    month = datetime.today().month
    if 3 <= month <= 8:
        return "https://richardslaw.com/in-office-consultation"
    else:
        return "https://richardslaw.com/virtual-consultation"

def generate_email_3(data):
    if data["status"] != "READY_FOR_AUTOMATION":
        return None, None

    booking_link = get_booking_link()

    subject = f"Next Steps Regarding Your Accident on {data['accident_date']}"

    body = f"""
Dear {data['client_name']},

We have reviewed the police report regarding your motor vehicle accident on {data['accident_date']} at {data['accident_location']}.

Based on the information involving {data['other_driver_name']} and your reported injuries ({', '.join(data.get('reported_injuries', []))}), we are prepared to move forward.

Please find your Retainer Agreement attached for review.

Your statute of limitations expires on {data['sol_date']}.

To schedule your consultation, please use the link below:
{booking_link}

Best regards,
Andrew Richards
"""
    return subject, body

In [ ]:
# MOJE DANE Z MAILTRAPA
mailtrap_user = "[REDACTED_FOR_SECURITY]"
mailtrap_pass = "[REDACTED_FOR_SECURITY]"

# Pobieramy dane z Twojej funkcji generującej
try:
    subject, body = generate_email_3(result_ready)
except Exception:
    # Backup na wypadek gdyby funkcja generate_email_3 nie była wczytana
    subject = "Legal Update"
    body = "Case is ready for processing."

if subject:
    import smtplib
    import time
    from email.message import EmailMessage

    msg = EmailMessage()
    msg['Subject'] = subject
    msg['From'] = "Legal AI Engine <legal-ai@swans-hackathon.com>"
    msg['To'] = f"{result_ready.get('client_name', 'Client')} <test@example.com>"
    msg.set_content(body)

    try:
        # Krótka pauza, żeby Mailtrap nas nie zablokował za tempo
        time.sleep(1)
        with smtplib.SMTP("sandbox.smtp.mailtrap.io", 2525) as server:
            server.starttls()
            server.login(mailtrap_user, mailtrap_pass)
            server.send_message(msg)
        print("✅ SUCCESS: Email wysłany do Sandboxa Mailtrap.")
        print("Sprawdź wynik tutaj: https://mailtrap.io/inboxes")
    except Exception as e:
        print(f"❌ STATUS: Serwer zajęty (Mailtrap limit). Spróbuj za moment. Błąd: {e}")
else:
    print("⚠️ STATUS: Wysyłka wstrzymana (Review Required).")

✅ SUCCESS: Email wysłany do Sandboxa Mailtrap.
Sprawdź wynik tutaj: https://mailtrap.io/inboxes


In [ ]:
from google.colab import userdata
password = userdata.get('MAILTRAP_PASS')